In [1]:
import json, re
import pandas as pd
import numpy as np
from get_electronic import get_electronic_dipole, get_electronic_quadrupole, get_stddev
from os import path
from pathlib import Path
from numpy import sign
import subprocess
from IPython.display import display

In [43]:
with open('output.json') as f:
    data = json.load(f)

In [3]:
with open('jacob_ladder.json') as f:
    ladder = json.load(f)

In [4]:
data

{'AlF': [{'name': 'AlF',
   'method': 'BLYP',
   'basis': 'aug-pc-4p',
   'method_type': 'dft',
   'method_level': 2,
   'quadrupoles': [-16.0501, -16.0501, -22.7295],
   'quadrupoles_off_diag': [-0.0, -0.0, -0.0],
   'dipoles': [-0.0, 0.0, -1.4687, 1.4687],
   'quadrupoles_elec': [-11.932880197700001,
    -11.932880197700001,
    -104.86651449752497],
   'dipoles_elec': [-0.0, 0.0, -28.71515488961, 28.71515488961],
   'std_dev': [0.736480580429654, 0.736480580429654, 1.7501494615108055],
   'spin_polarized': 0}],
 'Ar': [{'name': 'Ar',
   'method': 'B3LYP',
   'basis': 'aug-pc-4p',
   'method_type': 'dft',
   'method_level': 4,
   'quadrupoles': [-11.7543, -11.7543, -11.7543],
   'quadrupoles_off_diag': [-0.0, 0.0, -0.0],
   'dipoles': [0.0, 0.0, -0.0, 0.0],
   'quadrupoles_elec': [-8.739051701100001,
    -8.739051701100001,
    -8.739051701100001],
   'dipoles_elec': [0.0, 0.0, -0.0, 0.0],
   'std_dev': [0.6967803615798406, 0.6967803615798406, 0.6967803615798406],
   'spin_polarized'

In [5]:
def search(key, value, molecule):
    result = []
    for d in data[molecule]:
        if d[key] == value:
            result.append(d)
    return result

In [6]:
def load():
    global data
    with open('output.json') as f:
        data = json.load(f)

In [7]:
def save(data):
    with open("output.json", 'w') as outfile:
        json.dump(data, outfile)
    result = subprocess.run(['sh', 'commit.sh'], stdout=subprocess.PIPE)
    print(result.stdout)
    print('File saved!')

In [8]:
def save_as(data, name):
    with open(name, 'w') as outfile:
        json.dump(data, outfile)
    print('File saved as {0}!'.format(name))

In [9]:
def retrieve(molecule, method, basis):
    d = data[molecule]
    for m in d:
        if m['method'] == method and m['basis'] == basis:
            return m
    return {}

In [10]:
def get_dft_species():
    result = []
    for v in data.values():
        for d in v:
            if d['method_type'] == 'dft':
                result.append(d['name'])
                break
    return result
def get_wft_species():
    result = []
    for v in data.values():
        for d in v:
            if d['method_type'] == 'wft':
                result.append(d['name'])
                break
    return result

In [16]:
def get_functionals():
    functionals = []
    for v in data.values():
        for n in v:
            if n['method_type'] == 'dft':
                functionals.append(n['method'])
    functionals = np.unique(functionals)
    return functionals

In [11]:
def delete(molecule, method, basis):
    for c, d in enumerate(data[molecule]):
        if d.get('method') == method and d.get('basis') == basis:
            data[molecule].pop(c)
    return

def pop(molecule, method, basis, key):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d.pop(key, None)
    return

def modify(molecule, method, basis, old, new):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d[old] = new
    return

def add(molecule, method, basis, key, value):
    for d in data[molecule]:
        if d.get('method') == method and d.get('basis') == basis:
            d[key] = value

In [12]:
from pymatgen.core.periodic_table import Element

In [13]:
def count_e(name):
    if name == 'CH2-t':
        return 8
    l = [a for a in re.split(r'([A-Z][a-z]*)', name) if a]
    c = 0
    e = 0
    prev = ''
    while c < len(l):
        if l[c].isnumeric():
            e -= Element(prev).Z
            e += Element(prev).Z * int(l[c])
        else:
            e += Element(l[c]).Z
            prev = l[c]
        c += 1
    return e

In [14]:
def get_leq20():
    leq_20 = []
    for k in data.keys():
        if count_e(k) <= 20:
            if data[k][0]['method_type'] == 'dft':
                leq_20.append(data[k][0])
    return leq_20

In [29]:
#find electron count less than 20:
leq_20 = get_leq20()

In [30]:
leq_20

[{'name': 'Ar',
  'method': 'B3LYP',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 4,
  'quadrupoles': [-11.7543, -11.7543, -11.7543],
  'quadrupoles_off_diag': [-0.0, 0.0, -0.0],
  'dipoles': [0.0, 0.0, -0.0, 0.0],
  'quadrupoles_elec': [-15.8099292555267,
   -15.8099292555267,
   -15.8099292555267],
  'dipoles_elec': [0.0, 0.0, -0.0, 0.0],
  'std_dev': [0.9371922978155639, 0.9371922978155639, 0.9371922978155639],
  'spin_polarized': 0},
 {'name': 'Be',
  'method': 'PBE',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-7.5237, -7.5237, -7.5237],
  'quadrupoles_off_diag': [-0.0, 0.0, 0.0],
  'dipoles': [0.0, 0.0, 0.0, 0.0],
  'quadrupoles_elec': [-10.1196298154553,
   -10.1196298154553,
   -10.1196298154553],
  'dipoles_elec': [0.0, 0.0, 0.0, 0.0],
  'std_dev': [1.5905682801639875, 1.5905682801639875, 1.5905682801639875],
  'spin_polarized': 0},
 {'name': 'BeH',
  'method': 'B3LYP',
  'basis': 'aug-pc-4p',
  'method_type': 'df

In [31]:
asymmetrics = []
symmetrics = []
for m in leq_20:
    if abs(m['quadrupoles'][0] - m['quadrupoles'][1]) > 0.001:
        asymmetrics.append(m)
    elif all([v == 0 for v in m['quadrupoles_off_diag']]):
        symmetrics.append(m)

In [32]:
[x['name'] for x in asymmetrics]

['BH2',
 'BH2F',
 'C2H3',
 'CH2BH',
 'CH2F',
 'CH2NH',
 'CH2-t',
 'CH3BH2',
 'CH3O',
 'H2CN',
 'H2O',
 'HCO',
 'HNO',
 'HO2',
 'N2H2',
 'N2H4',
 'NH2',
 'NH2F',
 'NH2OH',
 'OF',
 'OH',
 'SH']

In [33]:
[x['name'] for x in leq_20]

['Ar',
 'Be',
 'BeH',
 'BeH2',
 'BF',
 'BH2',
 'BH2F',
 'BH3',
 'BO',
 'C2H',
 'C2H2',
 'C2H3',
 'CH2BH',
 'CH2F',
 'CH2NH',
 'CH2-t',
 'CH3',
 'CH3BH2',
 'CH3O',
 'CH4',
 'CN',
 'CO',
 'F2',
 'H',
 'H2',
 'H2CN',
 'H2O',
 'HBO',
 'HCN',
 'HCO',
 'He',
 'HF',
 'HNC',
 'HNO',
 'HO2',
 'LiBH4',
 'Mg',
 'N',
 'N2',
 'N2H2',
 'N2H4',
 'Na',
 'Ne',
 'NH2',
 'NH2F',
 'NH2OH',
 'NH3',
 'NH3O',
 'O2',
 'OF',
 'OH',
 'P',
 'SH',
 'LiH']

In [34]:
[x['name'] for x in symmetrics]

['Ar',
 'Be',
 'BeH',
 'BeH2',
 'BF',
 'BH3',
 'BO',
 'C2H',
 'C2H2',
 'CH3',
 'CH4',
 'CN',
 'CO',
 'F2',
 'H',
 'H2',
 'HBO',
 'HCN',
 'He',
 'HF',
 'HNC',
 'LiBH4',
 'Mg',
 'N',
 'N2',
 'Na',
 'Ne',
 'NH3',
 'NH3O',
 'O2',
 'P',
 'LiH']

In [17]:
functionals_sorted = sorted(sorted(get_functionals(), key=lambda v: v.upper()), key=lambda x: ladder[x])
functionals_sorted

['Slater',
 'SPW92',
 'B97-D3',
 'BLYP',
 'BPBE',
 'mPW91',
 'PBE',
 'B97M-rV',
 'B97M-V',
 'M06-L',
 'mBEEF',
 'MGGA_MS2',
 'MN15-L',
 'revM06-L',
 'SCAN',
 'TPSS',
 'B3LYP',
 'B3LYP5',
 'M06-2X',
 'PBE0',
 'M08-HX',
 'MN15',
 'SCAN0',
 'CAM-B3LYP',
 'wB97X-D',
 'wB97X-V',
 'wB97M-V']

In [18]:
def make_dft_table(key, domain=data.keys()):
    """
    Makes a table of molecules in DOMAIN of desired KEY.
    """
    values = []
    index = pd.MultiIndex.from_product([domain], names=['molecule'])
    columns = pd.MultiIndex.from_product([functionals_sorted, ['XX','YY','ZZ']], names=['method', 'component'])
    for d in domain:
        value = data[d]
        temp = [[0, 0, 0] for i in range(len(functionals_sorted))]
        for i in value:
            if i['method_type'] == 'dft':
                method = i['method']
                ind = functionals_sorted.index(method)
                temp[ind] = i[key]
        values.append(np.ravel(temp).tolist())
    df = pd.DataFrame(values, index=index, columns=columns)
    return df
              

In [19]:
quadrupoles_table = make_dft_table('quadrupoles')

In [20]:
make_dft_table('std_dev').loc['He']

method       Slater                        SPW92                      B97-D3  \
component        XX        YY        ZZ       XX       YY       ZZ        XX   
molecule                                                                       
He         0.665192  0.665192  0.665192  0.65497  0.65497  0.65497  0.641059   

method                             BLYP  ... CAM-B3LYP   wB97X-D            \
component        YY        ZZ        XX  ...        ZZ        XX        YY   
molecule                                 ...                                 
He         0.641059  0.641059  0.647148  ...  0.645538  0.635556  0.635556   

method                wB97X-V                       wB97M-V            \
component        ZZ        XX        YY        ZZ        XX        YY   
molecule                                                                
He         0.635556  0.642392  0.642392  0.642392  0.643086  0.643086   

method               
component        ZZ  
molecule             
He         0.643086  

[1 rows x 81 columns]

In [21]:
make_dft_table('std_dev').loc['BeH2']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
BeH2       0.947195  0.947195  1.893168   0.0  0.0  0.0  0.911522  0.911522   

method                  BLYP  ... CAM-B3LYP wB97X-D           wB97X-V       \
component       ZZ        XX  ...        ZZ      XX   YY   ZZ      XX   YY   
molecule                      ...                                            
BeH2       1.89122  0.915632  ...  1.891089     0.0  0.0  0.0     0.0  0.0   

method         wB97M-V            
component   ZZ      XX   YY   ZZ  
molecule                          
BeH2       0.0     0.0  0.0  0.0  

[1 rows x 81 columns]

In [22]:
make_dft_table('std_dev').loc['Ne']

method      Slater                      SPW92                      B97-D3  \
component       XX       YY       ZZ       XX       YY       ZZ        XX   
molecule                                                                    
Ne         0.57839  0.57839  0.57839  0.57307  0.57307  0.57307  0.567968   

method                             BLYP  ... CAM-B3LYP   wB97X-D            \
component        YY        ZZ        XX  ...        ZZ        XX        YY   
molecule                                 ...                                 
Ne         0.567968  0.567968  0.573919  ...   0.57041  0.565304  0.565304   

method                wB97X-V                       wB97M-V            \
component        ZZ        XX        YY        ZZ        XX        YY   
molecule                                                                
Ne         0.565304  0.569412  0.569412  0.569412  0.569934  0.569934   

method               
component        ZZ  
molecule             
Ne         0.569934  

[1 rows x 81 columns]

In [23]:
make_dft_table('std_dev').loc['Be']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
Be         1.222163  1.222163  1.222163   0.0  0.0  0.0  1.188805  1.188805   

method              BLYP  ... CAM-B3LYP wB97X-D           wB97X-V            \
component        ZZ   XX  ...        ZZ      XX   YY   ZZ      XX   YY   ZZ   
molecule                  ...                                                 
Be         1.188805  0.0  ...  1.183775     0.0  0.0  0.0     0.0  0.0  0.0   

method    wB97M-V            
component      XX   YY   ZZ  
molecule                     
Be            0.0  0.0  0.0  

[1 rows x 81 columns]

In [24]:
make_dft_table('std_dev').loc['N']

method       Slater                     SPW92              B97-D3            \
component        XX        YY        ZZ    XX   YY   ZZ        XX        YY   
molecule                                                                      
N          0.784452  0.784452  0.784452   0.0  0.0  0.0  0.765283  0.765283   

method                   BLYP  ... CAM-B3LYP   wB97X-D                      \
component        ZZ        XX  ...        ZZ        XX        YY        ZZ   
molecule                       ...                                           
N          0.765283  0.775054  ...  0.770291  0.764623  0.764623  0.764623   

method      wB97X-V                       wB97M-V                      
component        XX        YY        ZZ        XX        YY        ZZ  
molecule                                                               
N          0.770153  0.770153  0.770153  0.773229  0.773229  0.773229  

[1 rows x 81 columns]

In [25]:
wft_method = ['hf', 'MP2', 'CCSD', 'CCSD\(T\)']
wft_basis = ['aug-cc-pcV5Zp', 'aug-cc-pcVQZp', 'aug-cc-pcVTZp', 'CBS']

In [26]:
def make_table_wft_extrap(domain=data.keys()):
    """
    Makes a table of DOMAIN of wft method of desired KEY.
    """
    values = np.array([])
    index = pd.MultiIndex.from_product([domain, ['5Z', 'QZ', 'TZ', 'CBS']], names=['molecule', 'basis'])
    columns = pd.MultiIndex.from_product([wft_method, ['XX','YY','ZZ']], names=['method', 'component'])
    for d in domain:
        m_name = d
        for b in wft_basis:
            temp = [[0, 0, 0] for k in range(len(wft_method))]
            info = search('basis', b, m_name)
            if info != []:
                temp[0] = info[0]['std_dev_hf']
                temp[1] = info[0]['std_dev_mp2']
                temp[2] = info[0]['std_dev_ccsd']
                temp[3] = info[0]['std_dev_ccsdt']
            temp = np.ravel(temp)
            values = np.vstack((values, temp)) if values.size else temp
    df = pd.DataFrame(values, index=index, columns=columns)
    return df

In [44]:
wft_table = make_table_wft_extrap(get_wft_species())

KeyError: 'std_dev_hf'

In [28]:
wft_table.loc['HBO']

method           hf                           MP2                      \
component        XX        YY        ZZ        XX        YY        ZZ   
basis                                                                   
5Z         0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
QZ         0.761014  0.761014  1.596252  0.766903  0.766903  1.598887   
TZ         0.761262  0.761262  1.596361  0.768257  0.768257  1.599782   
CBS        0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   

method         CCSD                     CCSD\(T\)                      
component        XX        YY        ZZ        XX        YY        ZZ  
basis                                                                  
5Z         0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
QZ         0.763161  0.763161  1.596700  0.764286  0.764286  1.597599  
TZ         0.764681  0.764681  1.597664  0.765833  0.765833  1.598608  
CBS        0.000000  0.000000  0.000000  0.000000  0.000000  0.000000

In [44]:
wft_table[wft_table.index.get_level_values('basis') == '5Z']

method               hf                          MP2                        \
component            XX       YY       ZZ         XX         YY         ZZ   
molecule basis                                                               
AlF      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
Ar       5Z    -11.6741 -11.6741 -11.6741 -11.656129 -11.656129 -11.656129   
Be       5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
BeH      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
BeH2     5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   
...                 ...      ...      ...        ...        ...        ...   
HCl      5Z    -14.0726 -14.0726 -10.2180 -14.012628 -14.012628 -10.243776   
NH       5Z     -6.2690  -6.2690  -5.4578  -6.372837  -5.470655   0.836409   
PH       5Z    -14.4015 -14.4015 -15.4878 -14.363615 -15.049447  -0.236188   
SiO      5Z    -16.0826 -16.0826 -25.3663 -16.569470 -16.569470 -23.618388   
LiH      5Z      0.0000   0.0000   0.0000   0.000000   0.000000   0.000000   

method               CCSD                        CCSD\(T\)             \
component              XX         YY         ZZ         XX         YY   
molecule basis                                                          
AlF      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
Ar       5Z    -11.624722 -11.624722 -11.624722 -11.638576 -11.638576   
Be       5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
BeH      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
BeH2     5Z      0.000000   0.000000   0.000000   0.000000   0.000000   
...                   ...        ...        ...        ...        ...   
HCl      5Z    -13.947865 -13.947865 -10.236849 -13.963534 -13.963534   
NH       5Z     -6.347012  -5.482626   0.822085  -6.365237  -5.506568   
PH       5Z    -14.302954 -15.020933  -0.233700 -14.315261 -14.995915   
SiO      5Z    -16.259103 -16.259103 -24.055255 -16.382039 -16.382039   
LiH      5Z      0.000000   0.000000   0.000000   0.000000   0.000000   

method                     
component              ZZ  
molecule basis             
AlF      5Z      0.000000  
Ar       5Z    -11.638576  
Be       5Z      0.000000  
BeH      5Z      0.000000  
BeH2     5Z      0.000000  
...                   ...  
HCl      5Z    -10.266440  
NH       5Z      0.814889  
PH       5Z     -0.229194  
SiO      5Z    -23.702654  
LiH      5Z      0.000000  

[73 rows x 12 columns]

In [340]:
data['Be']

[{'name': 'Be',
  'method': 'PBE',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-7.5237, -7.5237, -7.5237],
  'quadrupoles_off_diag': [-0.0, 0.0, 0.0],
  'dipoles': [0.0, 0.0, 0.0, 0.0],
  'quadrupoles_elec': [-10.1196298154553,
   -10.1196298154553,
   -10.1196298154553],
  'dipoles_elec': [0.0, 0.0, 0.0, 0.0],
  'std_dev': [1.5905682801639875, 1.5905682801639875, 1.5905682801639875],
  'spin_polarized': 0},
 {'name': 'Be',
  'method': 'B97-D3',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-7.6035, -7.6035, -7.6035],
  'quadrupoles_off_diag': [0.0, 0.0, -0.0],
  'dipoles': [0.0, -0.0, -0.0, 0.0],
  'quadrupoles_elec': [-10.2269635022415,
   -10.2269635022415,
   -10.2269635022415],
  'dipoles_elec': [0.0, -0.0, -0.0, 0.0],
  'std_dev': [1.5989811992516907, 1.5989811992516907, 1.5989811992516907],
  'spin_polarized': 0},
 {'name': 'Be',
  'method': 'CAM-B3LYP',
  'basis': 'aug-pc-4p',
  'method_type': '

In [341]:
save(data)

File saved!


In [45]:
data['BeH']

[{'name': 'BeH',
  'method': 'B3LYP',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 4,
  'quadrupoles': [-6.7621, -6.7621, -10.1264],
  'quadrupoles_off_diag': [0.0, 0.0, 0.0],
  'dipoles': [-0.0, 0.0, 0.3051, 0.3051],
  'quadrupoles_elec': [-9.0952521731449, -9.0952521731449, -20.8306479857616],
  'dipoles_elec': [-0.0, 0.0, -5.25036441547, 5.25036441547],
  'std_dev': [1.3487217780658025, 1.3487217780658025, 1.75027898843098],
  'spin_polarized': 1},
 {'name': 'BeH',
  'method': 'B3LYP5',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 4,
  'quadrupoles': [-6.7758, -6.7758, -10.1462],
  'quadrupoles_off_diag': [0.0, 0.0, -0.0],
  'dipoles': [0.0, -0.0, 0.3008, 0.3008],
  'quadrupoles_elec': [-9.1136791344102, -9.1136791344102, -20.8572796524078],
  'dipoles_elec': [0.0, -0.0, -5.25205616576, 5.25205616576],
  'std_dev': [1.3500873404643272, 1.3500873404643272, 1.7515970346371494],
  'spin_polarized': 1},
 {'name': 'BeH',
  'method': 'BLYP',
  'ba

In [162]:
quadrupoles_table.loc['CO', 'Slater']

component,XX,YY,ZZ
molecule,,,
CO,-10.6552,-10.6552,-12.3091


In [163]:
data['PH']

[{'name': 'PH',
  'method': 'ccsdT',
  'basis': 'aug-cc-pcV5Zp',
  'method_type': 'wft',
  'quadrupoles_ccsdt': [-14.31526059084879,
   -14.995914878833943,
   -0.22919373704709678],
  'quadrupoles_mp2': [-14.363614551379564,
   -15.049447219275471,
   -0.23618791232231587],
  'quadrupoles_ccsd': [-14.30295353295378,
   -15.020932505427043,
   -0.23369959997213757],
  'quadrupoles_hf': [-14.4015, -14.4015, -15.4878],
  'dipoles_ccsdt': [-0.0, -0.0, -0.0, -0.0],
  'dipoles_mp2': [-0.0, -0.0, -0.0, -0.0],
  'dipoles_ccsd': [-0.0, -0.0, -0.0, -0.0],
  'dipoles_hf': [0.0, -0.0, -0.5129, 0.5129],
  'spin_polarized': 1,
  'dipoles_elec_hf': [0.0, -0.0, -40.518235085869996, 40.518235085869996],
  'dipoles_elec_mp2': [-0.0, -0.0, -40.31644468499999, 40.31644468499999],
  'dipoles_elec_ccsd': [-0.0, -0.0, -40.31644468499999, 40.31644468499999],
  'dipoles_elec_ccsdt': [-0.0, -0.0, -40.31644468499999, 40.31644468499999],
  'quadrupoles_elec_hf': [-10.707184015500001,
   -10.707184015500001,
   -

In [164]:
#this cell block is used to modify dipole values
modify('PH', 'ccsdT', 'aug-cc-pcVQZp', 'dipoles_ccsd', [0, 0, -0.429, 0.429])

In [158]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 2fd98b4] save Tue, Aug  4, 2020  4:40:42 PM\n 2 files changed, 96 insertions(+), 479 deletions(-)\n'
File saved!


In [159]:
#use this cell block for electronic component calculation

input_path = Path.home() / "summer/research/geometries/geometries"
ang_to_bohr = 1.88973
ang2_to_bohr2 = 3.571079
debye_to_e = 0.3934303
ea02_to_debye_ang = 1.345033669
debye_ang_to_ea02 = 0.743477

def get_electronic_dipole(string, dipole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    x = y = z = 0
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        if len(line) == 1:
            line += [0, 0, 0]
        x += charge * float(line[1])
        y += charge * float(line[2])
        z += charge * float(line[3])
    x_elec = dipole[0] * debye_to_e - x * ang_to_bohr
    y_elec = dipole[1] * debye_to_e - y * ang_to_bohr
    z_elec = dipole[2] * debye_to_e - z * ang_to_bohr
    total_elec = np.sqrt(x_elec ** 2 + y_elec ** 2 + z_elec ** 2)
    return [x_elec, y_elec, z_elec, total_elec]

def get_electronic_quadrupole(string, quadrupole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    xx = yy = zz = 0
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        if len(line) == 1:
            line += [0, 0, 0]
        xx += charge * float(line[1]) ** 2
        yy += charge * float(line[2]) ** 2
        zz += charge * float(line[3]) ** 2
    xx_elec = quadrupole[0] * debye_ang_to_ea02 - xx * ang2_to_bohr2
    yy_elec = quadrupole[1] * debye_ang_to_ea02 - yy * ang2_to_bohr2
    zz_elec = quadrupole[2] * debye_ang_to_ea02 - zz * ang2_to_bohr2
    return [xx_elec, yy_elec, zz_elec]

def get_stddev(string, dipole, quadrupole):
    info = re.search('molecule\s*\d\s\d\s([\s\S]*?)\$end', string).group(1)
    tote = int(re.search('molecule\s*(\d\s\d\s)', string).group(1).splitlines()[0].split()[0])
    for l in info.splitlines():
        line = l.split()
        charge = Element(re.sub('\d+', '', line[0])).Z
        tote += Element(re.sub('\d+', '', line[0])).Z
    xx_std = np.sqrt((quadrupole[0]/-tote) - (dipole[0]/-tote) ** 2)
    yy_std = np.sqrt((quadrupole[1]/-tote) - (dipole[1]/-tote) ** 2)
    zz_std = np.sqrt((quadrupole[2]/-tote) - (dipole[2]/-tote) ** 2)
    return [xx_std, yy_std, zz_std]

def get_electronic_for_wft(molecule, method, basis):
    filename = input_path / "{}.xyz".format(molecule)
    with open(filename, 'r') as file:
        string = file.read()
    d = retrieve(molecule, method, basis)
    if d == {}:
        print('WFT method has not been calculated for {0} of basis {1}!'.format(molecule, basis))
        return
    add(molecule, method, basis, 'dipoles_elec_hf', get_electronic_dipole(string, d['dipoles_hf']))
    add(molecule, method, basis, 'dipoles_elec_mp2', get_electronic_dipole(string, d['dipoles_mp2']))
    add(molecule, method, basis, 'dipoles_elec_ccsd', get_electronic_dipole(string, d['dipoles_ccsd']))
    add(molecule, method, basis, 'dipoles_elec_ccsdt', get_electronic_dipole(string, d['dipoles_ccsdt']))
    add(molecule, method, basis, 'quadrupoles_elec_hf', get_electronic_quadrupole(string, d['quadrupoles_hf']))
    add(molecule, method, basis, 'quadrupoles_elec_mp2', get_electronic_quadrupole(string, d['quadrupoles_mp2']))
    add(molecule, method, basis, 'quadrupoles_elec_ccsd', get_electronic_quadrupole(string, d['quadrupoles_ccsd']))
    add(molecule, method, basis, 'quadrupoles_elec_ccsdt', get_electronic_quadrupole(string, d['quadrupoles_ccsdt']))
    add(molecule, method, basis, 'std_dev_hf', get_stddev(string, d['dipoles_elec_hf'], d['quadrupoles_elec_hf']))
    add(molecule, method, basis, 'std_dev_mp2', get_stddev(string, d['dipoles_elec_mp2'], d['quadrupoles_elec_mp2']))
    add(molecule, method, basis, 'std_dev_ccsd', get_stddev(string, d['dipoles_elec_ccsd'], d['quadrupoles_elec_ccsd']))
    add(molecule, method, basis, 'std_dev_ccsdt', get_stddev(string, d['dipoles_elec_ccsdt'], d['quadrupoles_elec_ccsdt']))
    print('Electronic part successfully calculated for {0} of basis {1}!'.format(molecule, basis))
    
def get_electronic_for_dft(molecule, method, basis):
    filename = input_path / "{}.xyz".format(molecule)
    with open(filename, 'r') as file:
        string = file.read()
    d = retrieve(molecule, method, basis)
    if d == {}:
        print('DFT method has not been calculated for {0} of method {1}!'.format(molecule, method))
        return
    add(molecule, method, basis, 'dipoles_elec', get_electronic_dipole(string, d['dipoles']))
    add(molecule, method, basis, 'quadrupoles_elec', get_electronic_quadrupole(string, d['quadrupoles']))
    add(molecule, method, basis, 'std_dev', get_stddev(string, d['dipoles_elec'], d['quadrupoles_elec']))
    print('Electronic part successfully calculated for {0} of method {1}!'.format(molecule, method))

In [432]:
get_electronic_for_wft('HBO', 'ccsdT', 'aug-cc-pcVTZp')

Electronic part successfully calculated!


In [161]:
mols = data.keys()

In [162]:
bases = ['aug-cc-pcV5Zp', 'aug-cc-pcVQZp', 'aug-cc-pcVTZp']
for m in mols:
    for b in bases:
        get_electronic_for_wft(m, 'ccsdT', b)

WFT method has not been calculated for AlF of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for AlF of basis aug-cc-pcVQZp!
Electronic part successfully calculated for AlF of basis aug-cc-pcVTZp!
Electronic part successfully calculated for Ar of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for Ar of basis aug-cc-pcVQZp!
Electronic part successfully calculated for Ar of basis aug-cc-pcVTZp!
WFT method has not been calculated for Be of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for Be of basis aug-cc-pcVQZp!
Electronic part successfully calculated for Be of basis aug-cc-pcVTZp!
WFT method has not been calculated for BeH of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for BeH of basis aug-cc-pcVQZp!
Electronic part successfully calculated for BeH of basis aug-cc-pcVTZp!
WFT method has not been calculated for BeH2 of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for BeH2 of basis aug-cc-pcVQZp!
Electronic part 

Electronic part successfully calculated for NH2 of basis aug-cc-pcVQZp!
Electronic part successfully calculated for NH2 of basis aug-cc-pcVTZp!
WFT method has not been calculated for NH3 of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for NH3 of basis aug-cc-pcVQZp!
Electronic part successfully calculated for NH3 of basis aug-cc-pcVTZp!
WFT method has not been calculated for NH3O of basis aug-cc-pcV5Zp!
WFT method has not been calculated for NH3O of basis aug-cc-pcVQZp!
Electronic part successfully calculated for NH3O of basis aug-cc-pcVTZp!
WFT method has not been calculated for O2 of basis aug-cc-pcV5Zp!
Electronic part successfully calculated for O2 of basis aug-cc-pcVQZp!
Electronic part successfully calculated for O2 of basis aug-cc-pcVTZp!
WFT method has not been calculated for O3 of basis aug-cc-pcV5Zp!
WFT method has not been calculated for O3 of basis aug-cc-pcVQZp!
WFT method has not been calculated for O3 of basis aug-cc-pcVTZp!
Electronic part successfully c

IndexError: list index out of range

In [33]:
functionals = get_functionals()
for m in mols:
    for f in functionals:
        get_electronic_for_dft(m, f, 'aug-pc-4p')

DFT method has not been calculated for AlF of method B3LYP!
DFT method has not been calculated for AlF of method B3LYP5!
DFT method has not been calculated for AlF of method B97-D3!
DFT method has not been calculated for AlF of method B97M-V!
DFT method has not been calculated for AlF of method B97M-rV!
Electronic part successfully calculated for AlF of method BLYP!
DFT method has not been calculated for AlF of method BPBE!
DFT method has not been calculated for AlF of method CAM-B3LYP!
DFT method has not been calculated for AlF of method M06-2X!
DFT method has not been calculated for AlF of method M06-L!
DFT method has not been calculated for AlF of method M08-HX!
DFT method has not been calculated for AlF of method MGGA_MS2!
DFT method has not been calculated for AlF of method MN15!
DFT method has not been calculated for AlF of method MN15-L!
DFT method has not been calculated for AlF of method PBE!
DFT method has not been calculated for AlF of method PBE0!
DFT method has not been ca

Electronic part successfully calculated for CO of method Slater!
Electronic part successfully calculated for CO of method TPSS!
DFT method has not been calculated for CO of method mBEEF!
Electronic part successfully calculated for CO of method mPW91!
Electronic part successfully calculated for CO of method revM06-L!
Electronic part successfully calculated for CO of method wB97M-V!
Electronic part successfully calculated for CO of method wB97X-D!
Electronic part successfully calculated for CO of method wB97X-V!
DFT method has not been calculated for CO2 of method B3LYP!
DFT method has not been calculated for CO2 of method B3LYP5!
Electronic part successfully calculated for CO2 of method B97-D3!
DFT method has not been calculated for CO2 of method B97M-V!
DFT method has not been calculated for CO2 of method B97M-rV!
DFT method has not been calculated for CO2 of method BLYP!
DFT method has not been calculated for CO2 of method BPBE!
Electronic part successfully calculated for CO2 of metho

Electronic part successfully calculated for N2 of method mPW91!
Electronic part successfully calculated for N2 of method revM06-L!
Electronic part successfully calculated for N2 of method wB97M-V!
Electronic part successfully calculated for N2 of method wB97X-D!
Electronic part successfully calculated for N2 of method wB97X-V!
Electronic part successfully calculated for N2H2 of method B3LYP!
Electronic part successfully calculated for N2H2 of method B3LYP5!
Electronic part successfully calculated for N2H2 of method B97-D3!
Electronic part successfully calculated for N2H2 of method B97M-V!
DFT method has not been calculated for N2H2 of method B97M-rV!
Electronic part successfully calculated for N2H2 of method BLYP!
Electronic part successfully calculated for N2H2 of method BPBE!
Electronic part successfully calculated for N2H2 of method CAM-B3LYP!
Electronic part successfully calculated for N2H2 of method M06-2X!
Electronic part successfully calculated for N2H2 of method M06-L!
Electron

DFT method has not been calculated for SiO of method revM06-L!
DFT method has not been calculated for SiO of method wB97M-V!
DFT method has not been calculated for SiO of method wB97X-D!
DFT method has not been calculated for SiO of method wB97X-V!
DFT method has not been calculated for LiH of method B3LYP!
DFT method has not been calculated for LiH of method B3LYP5!
DFT method has not been calculated for LiH of method B97-D3!
DFT method has not been calculated for LiH of method B97M-V!
DFT method has not been calculated for LiH of method B97M-rV!
Electronic part successfully calculated for LiH of method BLYP!
Electronic part successfully calculated for LiH of method BPBE!
Electronic part successfully calculated for LiH of method CAM-B3LYP!
DFT method has not been calculated for LiH of method M06-2X!
DFT method has not been calculated for LiH of method M06-L!
DFT method has not been calculated for LiH of method M08-HX!
DFT method has not been calculated for LiH of method MGGA_MS2!
DFT 

In [34]:
data['LiH']

[{'name': 'LiH',
  'method': 'PBE',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-5.6189, -5.6189, -11.6417],
  'quadrupoles_off_diag': [-0.0, 0.0, -0.0],
  'dipoles': [-0.0, -0.0, -5.6155, 5.6155],
  'quadrupoles_elec': [-4.1775229153, -4.1775229153, -17.73911130538479],
  'dipoles_elec': [-0.0, -0.0, -5.22323822665, 5.22323822665],
  'std_dev': [1.021949474692854, 1.021949474692854, 1.6521619860277266],
  'spin_polarized': 0},
 {'name': 'LiH',
  'method': 'BLYP',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-5.6429, -5.6429, -11.7088],
  'quadrupoles_off_diag': [-0.0, 0.0, -0.0],
  'dipoles': [0.0, 0.0, -5.6037, 5.6037],
  'quadrupoles_elec': [-4.195366363300001,
   -4.195366363300001,
   -17.78899861208479],
  'dipoles_elec': [0.0, 0.0, -5.2185957491099995, 5.2185957491099995],
  'std_dev': [1.0241296748092987, 1.0241296748092987, 1.6568466445274552],
  'spin_polarized': 0},
 {'name': 'LiH',
  'metho

In [35]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 6320558] save Mon, Aug  3, 2020  9:38:26 PM\n 672 files changed, 440695 insertions(+), 11108 deletions(-)\n create mode 100644 output/dft/20-07-28/BO_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SCAN0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SCAN_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BO_SPW92_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_PBE_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SCAN0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SCAN_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_SPW92_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/BS_Slater_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_PBE0_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_PBE_aug-pc-4p.out\n create mode 100644 output/dft/20-07-28/CH3_SCAN0_aug-pc-4p.out\n create mode 100644

In [492]:
wft_table.to_csv('wft_table_2.csv')

In [36]:
std_dev_dft = make_dft_table('std_dev', get_wft_species())

In [37]:
std_dev_dft.to_csv('dft_table.csv')

In [447]:
quadrupoles_table.loc['HCl']

method      Slater                      SPW92                   B97-D3       \
component       XX       YY       ZZ       XX       YY       ZZ     XX   YY   
molecule                                                                      
HCl       -14.4753 -14.4753 -10.7091 -14.1432 -14.1432 -10.4192    0.0  0.0   

method         BLYP  ... CAM-B3LYP wB97X-D           wB97X-V            \
component   ZZ   XX  ...        ZZ      XX   YY   ZZ      XX   YY   ZZ   
molecule             ...                                                 
HCl        0.0  0.0  ...       0.0     0.0  0.0  0.0     0.0  0.0  0.0   

method    wB97M-V            
component      XX   YY   ZZ  
molecule                     
HCl           0.0  0.0  0.0  

[1 rows x 81 columns]

In [488]:
data['HBO']

[{'name': 'HBO',
  'method': 'PBE',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-11.0978, -11.0978, -19.9502],
  'quadrupoles_off_diag': [0.0, 0.0, 0.0],
  'dipoles': [-0.0, 0.0, -2.5855, 2.5855],
  'quadrupoles_elec': [-8.2509590506, -8.2509590506, -199.033869166212],
  'dipoles_elec': [-0.0, 0.0, -47.803149380650005, 47.803149380650005],
  'std_dev': [0.7676940913541018, 0.7676940913541018, 1.599319175331429],
  'spin_polarized': 0},
 {'name': 'HBO',
  'method': 'B97-D3',
  'basis': 'aug-pc-4p',
  'method_type': 'dft',
  'method_level': 2,
  'quadrupoles': [-11.0084, -11.0084, -20.1871],
  'quadrupoles_off_diag': [0.0, -0.0, 0.0],
  'dipoles': [-0.0, -0.0, -2.6504, 2.6504],
  'quadrupoles_elec': [-8.1844922068, -8.1844922068, -199.209998867512],
  'dipoles_elec': [-0.0, -0.0, -47.82868300712, 47.82868300712],
  'std_dev': [0.7645957011388437, 0.7645957011388437, 1.599357437857018],
  'spin_polarized': 0},
 {'name': 'HBO',
  'method': 'CAM-B3

In [507]:
save(data)

b'\x1bSaving updates to GitHub...\x1b\n[master 8bc6862] save\n 1 file changed, 1 insertion(+), 1 deletion(-)\n'
File saved!


In [557]:
wft_table.loc[('Ar', 'CBS'), ('hf', 'XX')] = 0.694399

In [567]:
wft_table.index[wft_table.index.get_level_values('basis') == 'CBS']

MultiIndex([(   'Ar', 'CBS'),
            (   'Be', 'CBS'),
            (  'BeH', 'CBS'),
            ( 'BeH2', 'CBS'),
            (   'BF', 'CBS'),
            (  'BH3', 'CBS'),
            (   'BO', 'CBS'),
            (   'BS', 'CBS'),
            (  'C2H', 'CBS'),
            ( 'C2H2', 'CBS'),
            (  'CH3', 'CBS'),
            (  'CH4', 'CBS'),
            (   'CN', 'CBS'),
            (   'CO', 'CBS'),
            (   'F2', 'CBS'),
            (    'H', 'CBS'),
            (   'H2', 'CBS'),
            (  'H2O', 'CBS'),
            (  'HBO', 'CBS'),
            (  'HCN', 'CBS'),
            (   'He', 'CBS'),
            (   'HF', 'CBS'),
            (  'HNC', 'CBS'),
            ('LiBH4', 'CBS'),
            (   'Mg', 'CBS'),
            (    'N', 'CBS'),
            (   'N2', 'CBS'),
            (   'Na', 'CBS'),
            (   'Ne', 'CBS'),
            (  'NH3', 'CBS'),
            ( 'NH3O', 'CBS'),
            (   'O2', 'CBS'),
            (   'OF', 'CBS'),
          

In [579]:
#make extrapolation in this cell
def extrapolate(dataframe):
    molecules = wft_table.index[wft_table.index.get_level_values('basis') == 'CBS']
    columns = wft_table.columns
    for m in molecules:
        for c in columns:
            if c[0] == 'hf':
                dataframe.loc[m, c] = dataframe.loc[(m[0], '5Z'), c]
            if dataframe.loc[(m[0], '5Z'), c] != 0:
                dataframe.loc[m, c] = ((dataframe.loc[(m[0], '5Z'), c] - dataframe.loc[(m[0], '5Z'), ('hf', c[1])]) * 125
                                       - (dataframe.loc[(m[0], 'QZ'), c] - dataframe.loc[(m[0], 'QZ'), ('hf', c[1])]) * 64) \
                                       / (125 - 64) + dataframe.loc[(m[0], 'CBS'), ('hf', c[1])]
            else:
                dataframe.loc[m, c] = ((dataframe.loc[(m[0], 'QZ'), c] - dataframe.loc[(m[0], 'QZ'), ('hf', c[1])]) * 64
                                       - (dataframe.loc[(m[0], 'TZ'), c] - dataframe.loc[(m[0], 'TZ'), ('hf', c[1])]) * 27) \
                                       / (64 - 27) + dataframe.loc[(m[0], 'CBS'), ('hf', c[1])]
    display(dataframe)

In [580]:
extrapolate(wft_table)

method                hf                           MP2                      \
component             XX        YY        ZZ        XX        YY        ZZ   
molecule basis                                                               
Ar       5Z     0.694399  0.694399  0.694399  0.693865  0.693865  0.693865   
         QZ     0.694477  0.694477  0.694477  0.694383  0.694383  0.694383   
         TZ     0.694798  0.694798  0.694798  0.695628  0.695628  0.695628   
         CBS    0.694399  0.694399  0.694399  0.693402  0.693402  0.693402   
Be       5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
...                  ...       ...       ...       ...       ...       ...   
PH       CBS    0.818046  0.818046  1.038866  0.816495  0.835891  0.659105   
SiO      5Z     0.737226  0.737226  1.611787  0.748302  0.748302  1.637564   
         QZ     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
         TZ     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
         CBS    0.737226  0.737226  1.611787  0.759922  0.759922  1.664609   

method              CCSD                     CCSD\(T\)                      
component             XX        YY        ZZ        XX        YY        ZZ  
molecule basis                                                              
Ar       5Z     0.692929  0.692929  0.692929  0.693342  0.693342  0.693342  
         QZ     0.693444  0.693444  0.693444  0.693885  0.693885  0.693885  
         TZ     0.695001  0.695001  0.695001  0.695466  0.695466  0.695466  
         CBS    0.692470  0.692470  0.692470  0.692854  0.692854  0.692854  
Be       5Z     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
...                  ...       ...       ...       ...       ...       ...  
PH       CBS    0.814809  0.835207  0.659002  0.815134  0.834454  0.658838  
SiO      5Z     0.741260  0.741260  1.642066  0.744057  0.744057  1.638433  
         QZ     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
         TZ     0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  
         CBS    0.745493  0.745493  1.673833  0.751225  0.751225  1.666390  

[160 rows x 12 columns]

In [582]:
wft_table.loc['H2O']

method           hf                           MP2                      \
component        XX        YY        ZZ        XX        YY        ZZ   
basis                                                                   
5Z         0.749271  0.851185  0.808749  0.761483  0.858177  0.818400   
QZ         0.749390  0.851298  0.808814  0.761942  0.858719  0.818847   
TZ         0.749663  0.851517  0.809062  0.763070  0.859975  0.819952   
CBS        0.749271  0.851185  0.808749  0.761126  0.857728  0.818000   

method         CCSD                     CCSD\(T\)                      
component        XX        YY        ZZ        XX        YY        ZZ  
basis                                                                  
5Z         0.755686  0.856091  0.814369  0.757586  0.857536  0.816078  
QZ         0.756221  0.856623  0.814826  0.758117  0.858078  0.816541  
TZ         0.757602  0.858066  0.816155  0.759484  0.859548  0.817866  
CBS        0.755249  0.855653  0.813957  0.757153  0.857086  0.815660

In [38]:
data.keys()

dict_keys(['AlF', 'Ar', 'Be', 'BeH', 'BeH2', 'BF', 'BH2', 'BH2F', 'BH3', 'BO', 'BS', 'C2H', 'C2H2', 'C2H3', 'CH2BH', 'CH2F', 'CH2NH', 'CH2-t', 'CH3', 'CH3BH2', 'CH3O', 'CH4', 'ClF', 'CN', 'CO', 'CO2', 'F2', 'FCN', 'FCO', 'H', 'H2', 'H2CN', 'H2O', 'HBO', 'HCCF', 'HCN', 'HCO', 'He', 'HF', 'HNC', 'HNO', 'HNS', 'HO2', 'LiBH4', 'Mg', 'Mg2', 'N', 'N2', 'N2H2', 'N2H4', 'Na', 'Na2', 'NaCl', 'Ne', 'NH2', 'NH2Cl', 'NH2F', 'NH2OH', 'NH3', 'NH3O', 'O2', 'O3', 'OF', 'OH', 'P', 'S2H2', 'SF', 'SH', 'HCl', 'NH', 'PH', 'SiO', 'LiH'])

In [42]:
std_dev_dft.loc['NH']

method       Slater                         SPW92                      \
component        XX        YY        ZZ        XX        YY        ZZ   
molecule                                                                
NH         0.790495  0.790495  1.000869  0.780677  0.780677  0.989187   

method       B97-D3                         BLYP  ... CAM-B3LYP   wB97X-D  \
component        XX        YY        ZZ       XX  ...        ZZ        XX   
molecule                                          ...                       
NH         0.769918  0.769918  0.985776  0.77911  ...  0.985582  0.769121   

method                          wB97X-V                       wB97M-V  \
component        YY        ZZ        XX        YY        ZZ        XX   
molecule                                                                
NH         0.769121  0.980772  0.774492  0.774492  0.984266  0.777988   

method                         
component        YY        ZZ  
molecule                       
NH         0.777988  0.986403  

[1 rows x 81 columns]